In [1]:
from pathlib import Path
import os
import pandas as pd
import sklearn

base_folder = Path(
    r"C:\Users\bamsa\OneDrive\Desktop\NUS - AI\Copstone Project"
)

part3_folder = base_folder / "capstone_part3"
part3_folder.mkdir(exist_ok=True)
os.chdir(part3_folder)

raw_csv = (
    base_folder / "capstone_part2"
    / "data" / "Metro_Interstate_Traffic_Volume.csv"
)

df = pd.read_csv(raw_csv, keep_default_na=False)

print("작업 폴더:", Path.cwd())
print("데이터 크기:", df.shape)
print("scikit-learn 버전:", sklearn.__version__)

df.head()

작업 폴더: C:\Users\bamsa\OneDrive\Desktop\NUS - AI\Copstone Project\capstone_part3
데이터 크기: (48204, 9)
scikit-learn 버전: 1.7.2


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,None,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,None,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,None,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,None,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,None,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


In [2]:
import logging

logger = logging.getLogger("part3_preparation")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(
            part3_folder / "part3.log",
            mode="a",
            encoding="utf-8",
        ),
    ],
    force=True,
)

# 원본 데이터는 유지하고 복사본으로 작업합니다.
data = df.copy()

data["date_time"] = pd.to_datetime(
    data["date_time"],
    format="%Y-%m-%d %H:%M:%S",
    errors="raise",
)

# 모든 열의 값이 동일한 중복만 제거합니다.
before = len(data)
data = data.drop_duplicates().copy()

logger.warning(
    "Removed exact duplicate rows=%d",
    before - len(data),
)

data = data.sort_values("date_time").reset_index(drop=True)

# 고유한 시각을 기준으로 분리합니다.
timestamps = data["date_time"].drop_duplicates().sort_values()
train_end = timestamps.iloc[int(len(timestamps) * 0.70)]
test_start = timestamps.iloc[int(len(timestamps) * 0.85)]

train = data.loc[data["date_time"] < train_end].copy()

validation = data.loc[
    (data["date_time"] >= train_end)
    & (data["date_time"] < test_start)
].copy()

test = data.loc[data["date_time"] >= test_start].copy()

# 분리 구간이 겹치지 않는지 확인합니다.
assert train["date_time"].max() < validation["date_time"].min()
assert validation["date_time"].max() < test["date_time"].min()

for name, subset in [
    ("train", train),
    ("validation", validation),
    ("test", test),
]:
    logger.info(
        "%s | rows=%d | unique timestamps=%d | start=%s | end=%s",
        name,
        len(subset),
        subset["date_time"].nunique(),
        subset["date_time"].min(),
        subset["date_time"].max(),
    )

summary = pd.DataFrame([
    {
        "split": name,
        "rows": len(subset),
        "unique_hours": subset["date_time"].nunique(),
        "start": subset["date_time"].min(),
        "end": subset["date_time"].max(),
    }
    for name, subset in [
        ("train", train),
        ("validation", validation),
        ("test", test),
    ]
])

summary

2026-10-10 22:31:13,082 | WARNING | part3_preparation | Removed exact duplicate rows=17
2026-10-10 22:31:13,153 | INFO | part3_preparation | train | rows=33497 | unique timestamps=28402 | start=2012-10-02 09:00:00 | end=2017-05-10 04:00:00
2026-10-10 22:31:13,156 | INFO | part3_preparation | validation | rows=7274 | unique timestamps=6086 | start=2017-05-10 05:00:00 | end=2018-01-19 14:00:00
2026-10-10 22:31:13,158 | INFO | part3_preparation | test | rows=7416 | unique timestamps=6087 | start=2018-01-19 15:00:00 | end=2018-09-30 23:00:00


,split,rows,unique_hours,start,end
0,train,33497,28402,2012-10-02 09:00:00,2017-05-10 04:00:00
1,validation,7274,6086,2017-05-10 05:00:00,2018-01-19 14:00:00
2,test,7416,6087,2018-01-19 15:00:00,2018-09-30 23:00:00


In [3]:
import numpy as np
import pandas as pd

weather_columns = [
    "temp", "rain_1h", "snow_1h", "clouds_all"
]

# 이상값을 결측값으로 바꾸기
def mark_invalid_weather(frame):
    result = frame.copy()

    for col in weather_columns:
        result[col] = pd.to_numeric(
            result[col], errors="coerce"
        ).replace([np.inf, -np.inf], np.nan)

    result.loc[result["temp"] <= 0, "temp"] = np.nan

    result.loc[
        (result["rain_1h"] < 0) | (result["rain_1h"] > 9000),
        "rain_1h"
    ] = np.nan

    result.loc[result["snow_1h"] < 0, "snow_1h"] = np.nan

    result.loc[
        ~result["clouds_all"].between(0, 100),
        "clouds_all"
    ] = np.nan

    return result


# 중앙값은 학습 데이터에서만 계산
train_checked = mark_invalid_weather(train)

monthly_medians = train_checked.groupby(
    train_checked["date_time"].dt.month
)[weather_columns].median()

overall_medians = train_checked[weather_columns].median()

if overall_medians.isna().any():
    raise ValueError("학습 데이터에서 중앙값을 계산할 수 없는 열이 있습니다.")


# 학습 데이터의 기준을 모든 구간에 적용
def clean_weather(frame, split_name):
    result = mark_invalid_weather(frame)
    month = result["date_time"].dt.month

    for col in weather_columns:
        missing_count = int(result[col].isna().sum())

        result[col] = (
            result[col]
            .fillna(month.map(monthly_medians[col]))
            .fillna(overall_medians[col])
        )

        if missing_count:
            logger.warning(
                "%s: filled %d invalid/missing values in %s",
                split_name, missing_count, col
            )

    logger.info(
        "%s: weather cleaning completed, rows=%d",
        split_name, len(result)
    )
    return result


train_clean = clean_weather(train, "train")
validation_clean = clean_weather(validation, "validation")
test_clean = clean_weather(test, "test")

# 결과 확인: 모든 칸이 0이면 결측값 처리 완료
pd.DataFrame({
    "train_missing": train_clean[weather_columns].isna().sum(),
    "validation_missing": validation_clean[weather_columns].isna().sum(),
    "test_missing": test_clean[weather_columns].isna().sum()
})

2026-10-10 22:36:28,809 | WARNING | part3_preparation | train: filled 10 invalid/missing values in temp
2026-10-10 22:36:28,813 | WARNING | part3_preparation | train: filled 1 invalid/missing values in rain_1h
2026-10-10 22:36:28,818 | INFO | part3_preparation | train: weather cleaning completed, rows=33497
2026-10-10 22:36:28,834 | INFO | part3_preparation | validation: weather cleaning completed, rows=7274
2026-10-10 22:36:28,848 | INFO | part3_preparation | test: weather cleaning completed, rows=7416


,train_missing,validation_missing,test_missing
temp,0,0,0
rain_1h,0,0,0
snow_1h,0,0,0
clouds_all,0,0,0


In [4]:
def create_features(frame):
    result = frame.copy()
    dt = result["date_time"]

    # 시간과 요일: 월요일=0, 일요일=6
    result["hour"] = dt.dt.hour
    result["day_of_week"] = dt.dt.dayofweek
    result["month"] = dt.dt.month
    result["is_weekend"] = (dt.dt.dayofweek >= 5).astype(int)

    # 순환 특성: 23시와 0시, 일요일과 월요일의 연결 표현
    result["hour_sin"] = np.sin(2 * np.pi * result["hour"] / 24)
    result["hour_cos"] = np.cos(2 * np.pi * result["hour"] / 24)
    result["dow_sin"] = np.sin(
        2 * np.pi * result["day_of_week"] / 7
    )
    result["dow_cos"] = np.cos(
        2 * np.pi * result["day_of_week"] / 7
    )

    # 날씨 문자열 통일
    for col in ["holiday", "weather_main", "weather_description"]:
        result[col] = (
            result[col].fillna("unknown")
            .astype(str).str.strip().str.lower()
        )

    # 데이터에 표시된 공휴일을 해당 날짜 전체에 적용
    holiday_mask = ~result["holiday"].isin(
        ["none", "unknown", "", "nan"]
    )
    holiday_dates = set(
        result.loc[holiday_mask, "date_time"].dt.date
    )
    result["is_holiday"] = (
        dt.dt.date.isin(holiday_dates).astype(int)
    )

    # 날씨 특성
    result["temp_c"] = result["temp"] - 273.15
    result["is_raining"] = (result["rain_1h"] > 0).astype(int)
    result["is_snowing"] = (result["snow_1h"] > 0).astype(int)

    # 관측 시정 거리가 없으므로 날씨 분류를 이용한 대리 지표
    result["is_low_visibility"] = (
        result["weather_main"]
        .isin(["mist", "fog", "haze", "smoke"])
        .astype(int)
    )

    return result


train_features = create_features(train_clean)
validation_features = create_features(validation_clean)
test_features = create_features(test_clean)

# 모델 입력에 사용할 열
numeric_features = [
    "hour", "day_of_week", "month",
    "is_weekend", "is_holiday",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
    "temp_c", "rain_1h", "snow_1h", "clouds_all",
    "is_raining", "is_snowing", "is_low_visibility"
]

categorical_features = ["weather_main"]
feature_columns = numeric_features + categorical_features

# 예측할 교통량은 입력 특성에서 제외
assert "traffic_volume" not in feature_columns

for name, frame in [
    ("train", train_features),
    ("validation", validation_features),
    ("test", test_features)
]:
    assert frame[feature_columns].isna().sum().sum() == 0
    logger.info(
        "%s: feature creation completed | rows=%d | input features=%d",
        name, len(frame), len(feature_columns)
    )

train_features[
    ["date_time", "hour", "day_of_week", "is_weekend",
     "is_holiday", "temp_c", "weather_main"]
].head()

2026-10-10 22:41:15,441 | INFO | part3_preparation | train: feature creation completed | rows=33497 | input features=17
2026-10-10 22:41:15,449 | INFO | part3_preparation | validation: feature creation completed | rows=7274 | input features=17
2026-10-10 22:41:15,456 | INFO | part3_preparation | test: feature creation completed | rows=7416 | input features=17


,date_time,hour,day_of_week,is_weekend,is_holiday,temp_c,weather_main
0,2012-10-02 09:00:00,9,1,0,0,15.13,clouds
1,2012-10-02 10:00:00,10,1,0,0,16.21,clouds
2,2012-10-02 11:00:00,11,1,0,0,16.43,clouds
3,2012-10-02 12:00:00,12,1,0,0,16.98,clouds
4,2012-10-02 13:00:00,13,1,0,0,17.99,clouds


In [5]:
# 학습 데이터에서만 사분위수 계산
q1, q2, q3 = train_features["traffic_volume"].quantile(
    [0.25, 0.50, 0.75]
).values

logger.info(
    "Training congestion thresholds: Q1=%.2f, Q2=%.2f, Q3=%.2f",
    q1, q2, q3
)

# 프로젝트에서 정한 악천후 대리 기준
# 실제 날씨의 강도나 사고 발생을 측정하는 기준은 아님
SEVERE_WEATHER = {"thunderstorm", "snow", "squall"}

def create_labels(frame):
    result = frame.copy()
    volume = result["traffic_volume"]

    if volume.isna().any() or (volume < 0).any():
        raise ValueError("교통량에 결측값 또는 음수가 있습니다.")

    result["congestion_category"] = np.select(
        [
            volume <= q1,
            volume <= q2,
            volume <= q3
        ],
        ["Low", "Medium", "High"],
        default="Severe"
    )

    high_congestion = result["congestion_category"].isin(
        ["High", "Severe"]
    )

    risky_weather = (
        result["weather_main"].isin(SEVERE_WEATHER)
        | (result["is_low_visibility"] == 1)
    )

    # 1: 높은 교통량과 악천후/낮은 시정 조건이 동시에 존재
    # 0: 위 조건을 충족하지 않음
    result["high_risk"] = (
        high_congestion & risky_weather
    ).astype(int)

    return result


train_features = create_labels(train_features)
validation_features = create_labels(validation_features)
test_features = create_labels(test_features)

# 정답에서 파생된 열은 모델 입력에서 제외
assert not {
    "traffic_volume", "congestion_category", "high_risk"
}.intersection(feature_columns)

label_summary = []

for name, frame in [
    ("train", train_features),
    ("validation", validation_features),
    ("test", test_features)
]:
    positive_count = int(frame["high_risk"].sum())

    label_summary.append({
        "split": name,
        "rows": len(frame),
        "proxy_0": len(frame) - positive_count,
        "proxy_1": positive_count,
        "proxy_1_percent": round(
            100 * frame["high_risk"].mean(), 2
        )
    })

    logger.info(
        "%s: labels created | proxy_1=%d",
        name, positive_count
    )

pd.DataFrame(label_summary)

2026-10-10 22:43:20,316 | INFO | part3_preparation | Training congestion thresholds: Q1=1160.00, Q2=3331.00, Q3=4924.00
2026-10-10 22:43:20,394 | INFO | part3_preparation | train: labels created | proxy_1=3688
2026-10-10 22:43:20,396 | INFO | part3_preparation | validation: labels created | proxy_1=903
2026-10-10 22:43:20,399 | INFO | part3_preparation | test: labels created | proxy_1=895


,split,rows,proxy_0,proxy_1,proxy_1_percent
0,train,33497,29809,3688,11.01
1,validation,7274,6371,903,12.41
2,test,7416,6521,895,12.07
